In [ ]:
import sys
import os
from util import Score_calculate as score
from transformers import T5Tokenizer, T5ForConditionalGeneration
import pandas as pd
from tqdm import tqdm
import torch

In [ ]:
# Define generation function
def generate_smiles_from_caption(caption, tokenizer, model, max_length=128):
    inputs = tokenizer(caption, return_tensors="pt", truncation=True).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            num_beams=5,
            early_stopping=True
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.strip()

In [ ]:
# SingMolT5 Model Generation and metrics computation

# Load model and tokenizer
tokenizer = T5Tokenizer.from_pretrained(R"YOUR_MODEL_PATH/SingMolT5_base", model_max_length=128)
model = T5ForConditionalGeneration.from_pretrained(R'YOUR_MODEL_PATH/SingMolT5_base')

# Load testing dataset
df = pd.read_csv("Dataset/MolGen_Test.csv")
captions = df["Caption"].tolist()
true_smiles = df["SMILES"].tolist()

# Set cuda
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Generate SMILES strings
ft_generated_smiles = []
for caption in tqdm(captions, desc="Generating SMILES"):
    pred = generate_smiles_from_caption(caption, tokenizer, model)
    ft_generated_smiles.append(pred)

# Compute evaluation metrics
with score.suppress_rdkit_warnings():
    results = score.evaluate_batch(true_smiles, ft_generated_smiles)

score.result_output(results)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Generating SMILES: 100%|██████████| 60/60 [00:20<00:00,  2.87it/s]



=== Evaluation Results (Average over Batch) ===
BLEU                  : 0.4539
Levenshtein Similarity: 0.5691
MACCS FTS             : 0.6233
RDK FTS               : 0.3267
Morgan FTS            : 0.4287
Validity              : 0.9833


In [ ]:
# MolT5 Model Generation and metrics computation

# Load model and tokenizer
tokenizer = T5Tokenizer.from_pretrained(R"YOUR_MODEL_PATH/MolT5base_Cap2Sm", model_max_length=128)
model = T5ForConditionalGeneration.from_pretrained(R'YOUR_MODEL_PATH/MolT5base_Cap2Sm')

# Load testing dataset
df = pd.read_csv("Dataset/MolGen_Test.csv")
captions = df["Caption"].tolist()
true_smiles = df["SMILES"].tolist()

# Set cuda
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Generate SMILES strings
mt5_generated_smiles = []
for caption in tqdm(captions, desc="Generating SMILES"):
    pred = generate_smiles_from_caption(caption, tokenizer, model)
    mt5_generated_smiles.append(pred)

# Compute evaluation metrics
with score.suppress_rdkit_warnings():
    results = score.evaluate_batch(true_smiles, mt5_generated_smiles)

score.result_output(results)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
/usr/local/anaconda3/envs/zyhEnv/lib/python3.8/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Generat


=== Evaluation Results (Average over Batch) ===
BLEU                  : 0.3833
Levenshtein Similarity: 0.4936
MACCS FTS             : 0.4220
RDK FTS               : 0.2362
Morgan FTS            : 0.2226
Validity              : 0.9167


In [ ]:
# T5 Model Generation and metrics computation

# Load model and tokenizer
tokenizer = T5Tokenizer.from_pretrained(R"YOUR_MODEL_PATH/T5base_Cap2Sm", model_max_length=128)
model = T5ForConditionalGeneration.from_pretrained(R'YOUR_MODEL_PATH/T5base_Cap2Sm')

# Load testing dataset
df = pd.read_csv("Dataset/MolGen_Test.csv")
captions = df["Caption"].tolist()
true_smiles = df["SMILES"].tolist()

# Set cuda
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Generate SMILES strings
t5_generated_smiles = []
for caption in tqdm(captions, desc="Generating SMILES"):
    pred = generate_smiles_from_caption(caption, tokenizer, model)
    t5_generated_smiles.append(pred)

# Compute evaluation metrics
with score.suppress_rdkit_warnings():
    results = score.evaluate_batch(true_smiles, t5_generated_smiles)

score.result_output(results)

Generating SMILES: 100%|██████████| 60/60 [00:31<00:00,  1.89it/s]


=== Evaluation Results (Average over Batch) ===
BLEU                  : 0.4084
Levenshtein Similarity: 0.4893
MACCS FTS             : 0.2068
RDK FTS               : 0.1419
Morgan FTS            : 0.1191
Validity              : 0.6167


In [13]:
print(f"Model is on device: {model.device}")

Model is on device: cuda:0


In [14]:
print(ft_generated_smiles)

['C1=CN=CC=C1C#CC2=CC=NC=C2', 'C1=CN=CC=C1C#CC2=CC=NC=C2', 'C1=CN=CC=C1C#CC2=CC=NC=C2', 'C1=CC(=CC=C1C(=O)O)C(=O)O', 'C1=CC(=CC=C1C#CC2=CC=C(C=C2)C(=O)O)C(=O)O', 'C1=CC(=CC=C1C#CC2=CC=C(C=C2)C(=O)O)C(=O)O', 'C1=CC(=CC=C1S)S', 'C1=CC(=CC=C1S)S', 'C1=CC(=CC=C1S)S', 'C1=CC(=CC=C1N)N', 'C1=CC(=CC=C1N)N', 'C1=CC(=CC=C1N)N', 'CSC1=CC=C(C=C1)C#CC2=CC=CC=C2', 'CSC1=CC=C(C=C1)C#CC2=CC=C(C=C2)SC', 'CSC1=CC=C(C=C1)SC', 'C1=CN=CC=C1C#CS', 'C1=CN=CC=C1C#CS', 'C1=CN=CC=C1S', 'CC(=O)SC1=CC=NC=C1', 'CC(=O)SC1=CC=NC=C1', 'CC(=O)SC1=CC=NC=C1', 'CNC1=CC=C(NC)C=C1', 'CNC1=CC=C(NC)C=C1', 'CNC1=CC=C(NC)C=C1', 'C1=CC(=CC=C1C#N)C#N', 'C1=CC(=CC=C1C#N)C#N', 'C1=CC(=CC=C1C#N)C#N', 'CNC1=CC2=C(C=C1)C=C(I)C=C2', 'C1=CC2=C(C=CC3=C2C=CC(=C3)I)C=C1I', 'C1=CC2=C(C=C1I)C=C(I)C=C2I', 'CSC1=CC=C(C=C1)C2=CC=C(C=C2)SC', 'CSC1=CC=C(C=C1)C2=CC=C(C=C2)SC', 'CSC1=CC=C(C=C1)C2=CC=C(C=C2)SC', 'CSC1=CC=C(SC)C=C1', 'CSC1=CC=C(SC)C=C1', 'CSC1=CC=C(SC)C=C1', 'C1=CC(=CC=C1N)N', 'C1=CC(=CC=C1N)N', 'C1=CC(=CC=C1N)N', 'C1=CC(=CC=C1C2=C

In [15]:
print(mt5_generated_smiles)

['CC1=C(C(=C(C(=C1C)C=O)N=C=O)C)C', 'C1=CC=NC(=C1)C2=CC=NC=C2', 'CC1=C(C(=C(C(=C1C2=CC=CC=C2)C)C)C=C)C=C CNN', 'C1=CC=C(C=C1)C2=CC=CC=C2', 'C1=CC=C(C=C1)C2=CC=CC=C2', 'C1=CC(=CC=C1C2=CC=C(C=C2)OC3=CC=C(C=C3)C(=O)O)C4=CC=CC=C4', 'C1=CC=C2C(=C1)C(=O)C3=CC=CC=C3S(=O)(=O)O', 'C[C@H]1[C@@H]([C@H]([C@H]([C@@H](O1)NS(=O)(=O)C2=CC=CC=C2)C(=O)O)N', 'C1=CC=C2C(=C1)C=CC(=O)CCC2=C1', 'C1=CC=C(C=C1)N', 'C1=CC=C(C=C1)C2=CC=C(C=C2)N', 'C1=CC=C(C=C1)C2=CC=C(C=C2)N', 'CSC1=CC=C(C=C1)SC2=CC=CC=C2C(=O)O', 'CSCCCCCCCCCCN1C2=CC=CC=C2C(=C1)SC', 'CSCCC[C@@H](C(=O)O)N', 'C1=CC(=CN=C1)C2=CC(=CC=C2)S(=O)(=O)N', 'C1=CN=CC=C1S(=O)(=O)C2=CC=C(C=C2)N3CCCCC3', 'C1=CC=C2C(=C1)C(=O)N(C2=O)C(=O)O', 'C1=CC=C2C(=C1)C(=O)C=C(N2)C3=CC=CC=N3', 'C1=CC(=CN=C1)C(=O)S', 'C1=CC=C(C=C1)S(=O)(=O)N', 'CNC1=CC=CC=C1', 'C1=CC=C(C=C1)CN', 'CNC1=CC=CC=C1', 'C1=CC=C(C=C1)C(=O)C2=CC=CC=C2', 'C1=CC=C(C=C1)C(=O)NC2=CC=CC=C2', 'C1=CC=C(C=C1)C(=O)C2=CC=CC=C2', 'C1=CC=C2C(=C1)C=CC(=C2)I', 'C1=CC=C2C(=C1)C=CC=C2I', 'C1=CC=C(C=C1)C(=O)NCCCC3=CC

In [16]:
print(t5_generated_smiles)

['CC1=CC=C(C=C1)C(=O)CC2=CC=CC=C2', 'C1=CC=C(C=C1)C(=O)C2=CC=C(C=C2)N(C)C(=O)O', 'C1=CC=C(C=C1)C(=O)NC2=C(C=CC(=C2)C(=C3C=CC(=[N+](C)C)CC(=O)O)C4=CC=CC=C4', 'CC1=CC=C(C=C1)CCCC(=O)CC2=CC=CC=C2', 'C1=CC=C(C=C1)CCC(=O)CCC2=CC=CC=C2', 'CC1=CC=C(C=C1)C(=O)C2=CC=CC=C2', 'C1=CC2=C(C=C1S(=O)(=O)O)C(=C(C=C2S(=O)(=O)O)S(=O)(=O)O', 'C1=CC(=CC=C1C(=O)O)S(=O)(=O)O', 'C1=CC=C2C(=C1)C(=O)C3=C(C2=O)C(=C(C=C3O)S(=O)(=O)O)S(=O)(=O)O)O', 'C1=CC=C(C=C1)C2=CC=CC=C2', 'C1=CC=C(C=C1)C2=CC=C(C=C2)N)N', 'C1=CC=C(C=C1)C2=CC(=CC=C2)N2C3=CC=CC=C3', 'C=CCSC1=CC=CC=C1C(=O)[C@H](CSC2=CC=CC=C2)C(=O)O', 'C[Sb](C)(C)C1=CC=C(C=C1)C(=O)O', 'C=CCSC1=CC=CC=C1C(=O)N(C(=O)N(C(=O)N(C(=O)N)C(=O)O)C)C', 'C1=CC(=CN=C1)C2=CC=CC=C2', 'C1=CC(=NC2=C(C=C1)N=C(S2)C(=O)O', 'C1=CC(=CN=C1)C2=CC=CC=C2S(=O)(=O)O', 'CC(=O)OC1=CC=C(C=C1)C(=O)[O-]', 'CSC1=C(C(=CC=C1)OC)C(=O)C2=CC=C(O2)C(=O)O', 'CCSC=NCC1=C(C(=C(C=N1)C)C(=O)[O-])C', 'CC1=C(C(=CC=C1)C)C', 'C[C@@H]1CCC(=O)C2=C(C3=C(C=C(C=C3C=C2', 'CN1C(=C(C2=C1C(=O)C3=CC=CC=C3N2) CNC2=O)C', 'C1